# Masked self-connected 2-unit GRU — v8

Trains a 2-unit GRU in which each hidden unit connects **to itself but not to the other**. The
hidden-to-hidden weight matrix `W_hh` is multiplied by a fixed binary mask on every forward pass, keeping
the diagonal (self-connections) and zeroing the off-diagonal (unit-to-unit connections).

The only subtlety is that PyTorch's GRU stacks three gate blocks in `W_hh` (reset, update, candidate), so at
2 units `W_hh` is 6x2 and the mask is the 2x2 identity tiled three times. The GRU cell is implemented by hand
so the mask is applied cleanly inside the recurrence, then trained exactly as before and put through the same
dynamics analysis, so it can be compared with the unmasked network.

Loads data from `./generated_trials_v8`.

## 1. Setup

In [ ]:
import math
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
import matplotlib.patches as mpatches
import torch, torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from scipy.optimize import minimize
from pathlib import Path

torch.manual_seed(0); np.random.seed(0)
device = torch.device("cpu")
DATA_DIR = Path("./generated_trials_v8"); OUT_DIR = Path("./masked_v8"); OUT_DIR.mkdir(exist_ok=True)
HIDDEN = 2; T_ON, T_OFF = 10, 20
CLASS_NAMES  = ["no det (0)", "det (1)", "right (2)", "left (3)"]
CLASS_COLORS = ["#cfcfcf", "#e0852e", "#c0392b", "#2e6ca4"]

## 2. Load data

In [ ]:
def load(split):
    d = np.load(DATA_DIR / f"{split}.npz", allow_pickle=True)
    out = {"X": d["X"].astype(np.float32), "y": d["y_4class"].astype(np.int64), "types": d["types"]}
    if "aud_int" in d:
        out["aud_int"] = d["aud_int"].astype(np.float32); out["vis_int"] = d["vis_int"].astype(np.float32)
    return out
train, test = load("train"), load("test")
T = test["X"].shape[2]
SUBTASKS = sorted(set(test["types"]))
CONFLICT = ["det_multisensory", "loc_conflict_audL_visR", "loc_conflict_audR_visL"]
NONCONF  = [s for s in SUBTASKS if s not in CONFLICT]
print("Train", train["X"].shape, " Test", test["X"].shape)

## 3. Masked GRU

A hand-written GRU cell (same equations as `torch.nn.GRU`) whose recurrent matrix is `W_hh * mask`.
`mask = I(2)` tiled three times, so every gate keeps unit->self and drops unit->other. Weights are
initialised with PyTorch's scheme (uniform +/- 1/sqrt(H)) for comparability.

In [ ]:
class MaskedGRU(nn.Module):
    def __init__(self, n_in=4, hidden=2, n_out=4):
        super().__init__(); self.H = hidden
        s = 1.0 / math.sqrt(hidden)
        self.weight_ih = nn.Parameter(torch.empty(3*hidden, n_in).uniform_(-s, s))
        self.weight_hh = nn.Parameter(torch.empty(3*hidden, hidden).uniform_(-s, s))
        self.bias_ih   = nn.Parameter(torch.empty(3*hidden).uniform_(-s, s))
        self.bias_hh   = nn.Parameter(torch.empty(3*hidden).uniform_(-s, s))
        self.readout   = nn.Linear(hidden, n_out)
        self.register_buffer("mask", torch.eye(hidden).repeat(3, 1))   # (3H, H)

    def masked_hh(self):
        return self.weight_hh * self.mask

    def forward(self, x):
        x = x.transpose(1, 2)                     # (B, C, T) -> (B, T, C)
        B, Tt, _ = x.shape; H = self.H
        Whh = self.masked_hh()
        Wir, Wiz, Win = self.weight_ih[:H], self.weight_ih[H:2*H], self.weight_ih[2*H:]
        Whr, Whz, Whn = Whh[:H], Whh[H:2*H], Whh[2*H:]
        bir, biz, bin_ = self.bias_ih[:H], self.bias_ih[H:2*H], self.bias_ih[2*H:]
        bhr, bhz, bhn = self.bias_hh[:H], self.bias_hh[H:2*H], self.bias_hh[2*H:]
        h = x.new_zeros(B, H); outs = []
        for t in range(Tt):
            xt = x[:, t, :]
            r = torch.sigmoid(xt @ Wir.T + bir + h @ Whr.T + bhr)
            z = torch.sigmoid(xt @ Wiz.T + biz + h @ Whz.T + bhz)
            n = torch.tanh(xt @ Win.T + bin_ + r * (h @ Whn.T + bhn))
            h = (1 - z) * n + z * h
            outs.append(self.readout(h))
        return torch.stack(outs, 1)               # (B, T, n_out)

## 4. Train (all seeds kept; no quality selection)

In [ ]:
def train_masked(seed, n_epochs=50, lr=1e-3, batch=64):
    torch.manual_seed(seed); np.random.seed(seed)
    model = MaskedGRU(4, HIDDEN, 4).to(device)
    loader = DataLoader(TensorDataset(torch.from_numpy(train["X"]), torch.from_numpy(train["y"])),
                        batch_size=batch, shuffle=True)
    opt = torch.optim.Adam(model.parameters(), lr=lr); loss_fn = nn.CrossEntropyLoss()
    for _ in range(n_epochs):
        model.train()
        for Xb, yb in loader:
            logits = model(Xb); B, Tt, C = logits.shape
            loss = loss_fn(logits.reshape(B*Tt, C), yb.unsqueeze(1).expand(B, Tt).reshape(B*Tt))
            opt.zero_grad(); loss.backward(); opt.step()
    return model

def evaluate(model):
    model.eval()
    with torch.no_grad(): pred = model(torch.from_numpy(test["X"]))[:, -1, :].argmax(-1).numpy()
    accs = {s: float((pred[test["types"] == s] == test["y"][test["types"] == s]).mean()) for s in SUBTASKS}
    return pred, accs, np.mean([accs[s] for s in NONCONF])

SEEDS = [0, 1, 2, 3, 4]
models = {}
for s in SEEDS:
    m = train_masked(s); models[s] = m
    _, _, nc = evaluate(m)
    off = (m.masked_hh().detach().numpy().reshape(3, HIDDEN, HIDDEN)[:, 0, 1]).max()
    print("seed %d  non-conflict %.3f   max masked off-diagonal = %.1e" % (s, nc, abs(off)))
DISPLAY_SEED = SEEDS[0]          # seed shown below; set to any value in SEEDS
model = models[DISPLAY_SEED]
_, accs, _ = evaluate(model)
print("\nseed %d per-subtask:" % DISPLAY_SEED)
for s in SUBTASKS: print("  %-26s %.3f" % (s, accs[s]))

## 5. Hand-coded numpy update (with the same mask), verified against the torch model

In [ ]:
def masked_params(model):
    Wih = model.weight_ih.detach().numpy(); Whh = model.masked_hh().detach().numpy()
    bih = model.bias_ih.detach().numpy();   bhh = model.bias_hh.detach().numpy()
    H = HIDDEN; sl = lambda M, i: M[i*H:(i+1)*H]
    return dict(Wir=sl(Wih,0), Wiz=sl(Wih,1), Win=sl(Wih,2),
                Whr=sl(Whh,0), Whz=sl(Whh,1), Whn=sl(Whh,2),
                bir=bih[:H], biz=bih[H:2*H], bin_=bih[2*H:3*H],
                bhr=bhh[:H], bhz=bhh[H:2*H], bhn=bhh[2*H:3*H], H=H)

def sigmoid(v): return 1.0 / (1.0 + np.exp(-v))
def gru_step(h, x, p):
    r = sigmoid(p["Wir"] @ x + p["bir"] + p["Whr"] @ h + p["bhr"])
    z = sigmoid(p["Wiz"] @ x + p["biz"] + p["Whz"] @ h + p["bhz"])
    n = np.tanh(p["Win"] @ x + p["bin_"] + r * (p["Whn"] @ h + p["bhn"]))
    return (1.0 - z) * n + z * h

P = masked_params(model)
Wro = model.readout.weight.detach().numpy(); bro = model.readout.bias.detach().numpy()
def readout_class(h): return int(np.argmax(Wro @ h + bro))

def torch_hidden(model, X):
    model.eval()
    x = torch.from_numpy(X).transpose(1, 2); B, Tt, _ = x.shape; H = HIDDEN
    Whh = model.masked_hh()
    Wir, Wiz, Win = model.weight_ih[:H], model.weight_ih[H:2*H], model.weight_ih[2*H:]
    Whr, Whz, Whn = Whh[:H], Whh[H:2*H], Whh[2*H:]
    bir, biz, bin_ = model.bias_ih[:H], model.bias_ih[H:2*H], model.bias_ih[2*H:]
    bhr, bhz, bhn = model.bias_hh[:H], model.bias_hh[H:2*H], model.bias_hh[2*H:]
    h = x.new_zeros(B, H); hs = []
    with torch.no_grad():
        for t in range(Tt):
            xt = x[:, t, :]
            r = torch.sigmoid(xt @ Wir.T + bir + h @ Whr.T + bhr)
            z = torch.sigmoid(xt @ Wiz.T + biz + h @ Whz.T + bhz)
            n = torch.tanh(xt @ Win.T + bin_ + r * (h @ Whn.T + bhn))
            h = (1 - z) * n + z * h; hs.append(h.clone())
    return torch.stack(hs, 1).numpy()

h_pt = torch_hidden(model, test["X"][:6]); err = 0.0
for b in range(6):
    h = np.zeros(HIDDEN)
    for t in range(T): h = gru_step(h, test["X"][b][:, t], P); err = max(err, abs(h - h_pt[b, t]).max())
print("max |numpy - torch| hidden =", err); assert err < 1e-4
print("OK: masked update reproduced in numpy.")

## 6. Dynamics: decision regions, trajectories, baseline fixed points

Same analysis as the unmasked 2-unit notebook, so the two can be compared directly.

In [ ]:
GRID = np.linspace(-1.05, 1.05, 350); GX, GY = np.meshgrid(GRID, GRID); _flat = np.stack([GX.ravel(), GY.ravel()], 1)
def draw_regions(ax, fill=True):
    reg = np.argmax(_flat @ Wro.T + bro, 1).reshape(GX.shape)
    if fill: ax.pcolormesh(GX, GY, reg, cmap=ListedColormap(CLASS_COLORS), alpha=0.20, shading="auto", vmin=0, vmax=3)
    ax.set_xlabel("hidden unit 1"); ax.set_ylabel("hidden unit 2"); ax.set_aspect("equal")
    ax.set_xlim(-1.05, 1.05); ax.set_ylim(-1.05, 1.05)
def label_regions(ax, fs=11):
    reg = np.argmax(_flat @ Wro.T + bro, 1)
    for cls in range(4):
        m = reg == cls
        if m.sum() < 50: continue
        ax.text(_flat[m,0].mean(), _flat[m,1].mean(), CLASS_NAMES[cls], ha="center", va="center",
                fontsize=fs, fontweight="bold",
                bbox=dict(boxstyle="round,pad=0.25", fc="white", ec=CLASS_COLORS[cls], lw=1.5, alpha=0.85), zorder=20)
def hidden_traj(idx):
    out = np.zeros((len(idx), T+1, HIDDEN))
    for k, i in enumerate(idx):
        h = np.zeros(HIDDEN)
        for t in range(T): h = gru_step(h, test["X"][i][:, t], P); out[k, t+1] = h
    return out

# average trajectory per subtask (Figure-1 style)
fig, ax = plt.subplots(figsize=(8.5, 8)); draw_regions(ax); label_regions(ax)
pred_all = model(torch.from_numpy(test["X"]))[:, -1, :].argmax(-1).detach().numpy()
cols = plt.cm.tab20(np.linspace(0, 1, 20)); ci = 0
for s in NONCONF:
    tr = hidden_traj(np.where(test["types"] == s)[0]).mean(0); c = cols[ci]; ci += 1
    ax.plot(tr[:,0], tr[:,1], color=c, lw=2, label=s); ax.scatter(*tr[-1], color=c, s=40, marker="*", edgecolor="k", lw=0.4, zorder=6)
for s in CONFLICT:
    labs = [("det",1),("no-det",0)] if s == "det_multisensory" else [("right",2),("left",3)]
    for nm, lb in labs:
        idx = np.where((test["types"] == s) & (pred_all == lb))[0]
        if len(idx)==0: continue
        tr = hidden_traj(idx).mean(0); c = cols[ci % 20]; ci += 1
        ax.plot(tr[:,0], tr[:,1], color=c, lw=2.4, ls="--", label="%s->%s" % (s, nm)); ax.scatter(*tr[-1], color=c, s=55, marker="*", edgecolor="k", lw=0.5, zorder=6)
ax.scatter(0,0,color="k",s=30,zorder=7)
ax.set_title("Masked GRU: average trajectory per subtask (seed %d)" % DISPLAY_SEED)
ax.legend(loc="center left", bbox_to_anchor=(1.01,0.5), fontsize=7.5, frameon=False)
plt.tight_layout(); plt.savefig(OUT_DIR/"masked_fig1.png", dpi=150, bbox_inches="tight"); plt.show()

## 7. Baseline fixed points and flow field

In [ ]:
def fixed_points(x, p, inits, tol=1e-10):
    def q(h): d = gru_step(h, x, p) - h; return 0.5*float(d@d)
    found=[]
    for h0 in inits:
        r = minimize(q, h0, method="L-BFGS-B", bounds=[(-1.2,1.2)]*p["H"], options=dict(maxiter=500))
        if r.fun < tol: found.append(r.x)
    uniq=[]
    for h in found:
        if not any(np.allclose(h,u,atol=1e-3) for u in uniq): uniq.append(h)
    return uniq
def jacobian(h, x, p, eps=1e-5):
    H=p["H"]; J=np.zeros((H,H)); f0=gru_step(h,x,p)
    for i in range(H):
        hp=h.copy(); hp[i]+=eps; J[:,i]=(gru_step(hp,x,p)-f0)/eps
    return J
def classify(h, x, p):
    ev=np.linalg.eigvals(jacobian(h,x,p)); m=np.abs(ev)
    return ("stable" if np.all(m<1) else "unstable" if np.all(m>1) else "saddle"), ev
def flow(ax, x, p, n=23, scale=8):
    g=np.linspace(-1,1,n); XX,YY=np.meshgrid(g,g); U=np.zeros_like(XX); V=np.zeros_like(YY)
    for i in range(n):
        for j in range(n):
            d=gru_step(np.array([XX[i,j],YY[i,j]]),x,p)-np.array([XX[i,j],YY[i,j]]); U[i,j],V[i,j]=d
    ax.quiver(XX,YY,U,V,color="0.35",alpha=0.6,scale=scale,width=0.003)

ends=[]
for i in range(0,len(test["X"]),7):
    h=np.zeros(HIDDEN)
    for t in range(T): h=gru_step(h,test["X"][i][:,t],P)
    ends.append(h)
inits = ends + [np.random.uniform(-1,1,HIDDEN) for _ in range(150)]
x0=np.zeros(4); fps=fixed_points(x0,P,inits)
fig,ax=plt.subplots(figsize=(7,7)); draw_regions(ax); label_regions(ax); flow(ax,x0,P)
print("Baseline fixed points (masked):")
for h in fps:
    k,ev=classify(h,x0,P); mk={"stable":"o","saddle":"X","unstable":"^"}[k]
    ax.scatter(*h,marker=mk,s=130,edgecolor="k",facecolor="white",linewidths=1.6,zorder=6)
    print("  h=[% .3f % .3f] %-8s -> %-10s |eig|=%s"%(h[0],h[1],k,CLASS_NAMES[readout_class(h)],np.round(np.abs(ev),3)))
ax.set_title("Masked GRU: baseline flow field and fixed points (seed %d)"%DISPLAY_SEED)
plt.tight_layout(); plt.savefig(OUT_DIR/"masked_flow.png",dpi=150,bbox_inches="tight"); plt.show()

## 8. Notes

- The mask keeps each unit's self-connection and forces the unit-to-unit weight to zero in all three
  gates, on every forward pass. The printout in cell 4 confirms the effective off-diagonal is ~0.
- Compare these figures against the unmasked 2-unit notebook (`04`/`05`). The interesting question is
  whether two units that cannot talk to each other can still route the tasks into the right regions, and
  how the fixed-point layout differs from the fully-connected case.
- Only the recurrent connectivity changed; data, sizes, training and readout are identical.